In [ ]:
import os
import json
from tqdm import tqdm
import httpx
from anthropic import Anthropic

# --- Claude client (with SSL verification disabled only if needed) ---
# ⚠️  Use verify=False only in trusted local or testing environments.
client = Anthropic(
    api_key=" ",
    http_client=httpx.Client(verify=False)
)

with open('valid_data.json', 'r') as vd:
    valid_json = json.load(vd)

# Collect results from LLM in dictionary
results = {}

# Use tqdm for progress bar
for entry in tqdm(valid_json, desc="Processing entries"):
    natural_s = valid_json[entry]["natural_s"]
    natural_p = valid_json[entry]["natural_p"]
    formal_s = valid_json[entry]["formal_s"]

    # Prompt (kept identical)
    prompt = f"""
            Translate the following Isabelle/HOL statement into a clear natural language question that conveys the exact same logical meaning.
            
            Generate only the natural language question written in LaTex, without any additional commentary or explanation.

            Formal Statement:
            {formal_s}

            Natural Language Statement:
            """

    # ---- Claude call ----
    response = client.messages.create(
       model="claude-3-5-sonnet-latest",
        max_tokens=500,
        temperature=0.0,
        system="You are a helpful assistant that translates between formal logic and natural language.",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    # Extract text parts from Claude’s response
    gpt_natural_s = "".join(
        blk.text for blk in response.content if getattr(blk, "type", None) == "text"
    ).strip()

    results[entry] = {
        "natural_s": natural_s,
        "natural_p": natural_p,
        "formal_s": formal_s,
        "gpt_natural_s": gpt_natural_s
    }

# Save results to a JSON file
with open('claude_translations_latex.json', 'w') as out_file:
    json.dump(results, out_file, indent=4)


Processing entries: 100%|██████████| 244/244 [06:06<00:00,  1.50s/it]


In [ ]:
import os
import json
from tqdm import tqdm
import httpx
from anthropic import Anthropic

# --- Claude 3.7 Sonnet client (disable SSL verification only if you must) ---
# ⚠️ In production, you should keep SSL verification enabled.
client = Anthropic(
    api_key=" ",
    http_client=httpx.Client(verify=False)
)

with open('valid_data.json', 'r') as vd:
    valid_json = json.load(vd)

# Collect results from LLM in dictionary
results = {}

# Use tqdm for progress bar
for entry in tqdm(valid_json, desc="Processing entries"):
    natural_s = valid_json[entry]["natural_s"]
    natural_p = valid_json[entry]["natural_p"]
    formal_s = valid_json[entry]["formal_s"]

    # Prompt (kept identical)
    prompt = f"""
            Translate the following Natural Language statement into a Isabelle/HOL Formal Statement that conveys the exact same logical meaning.
                
            Generate only the Isabelle/HOL Formal Statement, without any additional commentary or explanation.

            Natural Language Statement:
            {natural_s}

            Formal Statement:
            """

    # ---- Claude 3.7 Sonnet call ----
    response = client.messages.create(
        model="claude-3-7-sonnet-latest",
        max_tokens=500,
        temperature=0.0,
        system="You are a helpful assistant that translates between formal logic and natural language.",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    # Extract text parts from Claude’s response
    claude_formal_s = "".join(
        blk.text for blk in response.content if getattr(blk, "type", None) == "text"
    ).strip()

    results[entry] = {
        "natural_s": natural_s,
        "natural_p": natural_p,
        "formal_s": formal_s,
        "gpt_formal_s": claude_formal_s
    }

# Save results to a JSON file
with open('claude_translations_latex_1_formal.json', 'w') as out_file:
    json.dump(results, out_file, indent=4)


Processing entries: 100%|██████████| 244/244 [07:51<00:00,  1.93s/it]


In [ ]:
import os
import json
from tqdm import tqdm
import httpx
from anthropic import Anthropic

# --- Claude 3.7 Sonnet client (SSL verification disabled only if you truly need it) ---
# ⚠️  In production, keep SSL verification enabled for security.
client = Anthropic(
    api_key=" ",
    http_client=httpx.Client(verify=False)
)

with open('valid_data.json', 'r') as vd:
    valid_json = json.load(vd)

# valid_dic[data['problem_name']]={"natural_s":data["informal_statement"],
#                                  "natural_p":data["informal_proof"], 
#                                  "formal_s":data["formal_statement"]}

# Collect results from LLM in dictionary
results = {}

# Use tqdm for progress bar
for entry in tqdm(valid_json, desc="Processing entries"):
    natural_s = valid_json[entry]["natural_s"]
    natural_p = valid_json[entry]["natural_p"]
    formal_s = valid_json[entry]["formal_s"]

    # Prompt (unchanged)
    prompt = f"""
            Translate the following Isabelle/HOL statement into 10 clear natural language questions that conveys the exact same logical meaning.
            
            Generate a numbered list of 10 unique natural language questions that conveys the exact same logical meaning written in LaTex, without any additional commentary or explanation.

            Formal Statement:
            {formal_s}

            Natural Language Statements:
            1.
            """

    # ---- Claude 3.7 Sonnet call ----
    response = client.messages.create(
        model="claude-3-7-sonnet-latest",
        max_tokens=5000,
        temperature=0.0,
        system="You are a helpful assistant that translates between formal logic and natural language.",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    # Extract text parts from Claude’s response
    claude_output = "".join(
        blk.text for blk in response.content if getattr(blk, "type", None) == "text"
    ).strip()

    results[entry] = {
        "natural_s": natural_s,
        "natural_p": natural_p,
        "formal_s": formal_s,
        "gpt_natural_s": claude_output
    }

# Save results to a JSON file
with open('claude_translations_latex_10.json', 'w') as out_file:
    json.dump(results, out_file, indent=4)


Processing entries: 100%|██████████| 244/244 [34:48<00:00,  8.56s/it]


In [6]:
with open('claude_translations_latex_10.json', 'r') as out_file:
    my_json = json.load(out_file)


for entry in my_json:
    # Try to split based on "\item" after removing "\begin{enumerate}" and "\end{enumerate}"
    # Otherwise we need to split based on new lines
    if "\\begin{enumerate}" in my_json[entry]["gpt_natural_s"]:
        items = my_json[entry]["gpt_natural_s"].replace("\\begin{enumerate}", "").replace("\\end{enumerate}", "").split("\\item")
        # Remove any leading/trailing whitespace and filter out empty strings
        items = [item.strip() for item in items if item.strip()]
        my_json[entry]["gpt_natural_s_list"] = items
    else:
        items = my_json[entry]["gpt_natural_s"].split("\n")
        # Remove leading numbers after splitting on dot or parenthesis
        items = [item.split('.', 1)[-1] if '.' in item else item.split(')', 1)[-1] for item in items]
        
        items = [item.strip() for item in items if item.strip()]
        my_json[entry]["gpt_natural_s_list"] = items
    
   
with open('claude_translations_latex_10_formatted.json', 'w') as out_file:
    json.dump(my_json, out_file, indent=4)


In [ ]:
import os
import json
from tqdm import tqdm
import httpx
from anthropic import Anthropic

# --- Claude client (disable SSL verification only if you must) ---
client = Anthropic(
    api_key=" ",
    http_client=httpx.Client(verify=False)  # keep/ remove as needed
)

with open('claude_translations_latex_10_formatted.json', 'r') as vd:
    valid_json = json.load(vd)

# Collect results from LLM in dictionary
results = {}

# Use tqdm for progress bar
for entry in tqdm(valid_json, desc="Processing entries"):
    gpt_natural_s_list = valid_json[entry]["gpt_natural_s_list"]
    gpt_formal_s_list = []
    natural_s = valid_json[entry]["natural_s"]
    natural_p = valid_json[entry]["natural_p"]
    formal_s = valid_json[entry]["formal_s"]

    for i, gpt_natural_s in enumerate(gpt_natural_s_list):
        # Prompt (unchanged)
        prompt = f"""
                Translate the following Natural Language statement into a Isabelle/HOL Formal Statement that conveys the exact same logical meaning.
                
                Generate only the Isabelle/HOL Formal Statement, without any additional commentary or explanation.

                Natural Language Statement:
                {gpt_natural_s}

                Formal Statement:
                """

        # ---- Claude 3.7 Sonnet call ----
        response = client.messages.create(
            model="claude-3-7-sonnet-latest",
            max_tokens=500,
            temperature=0.0,
            system="You are a helpful assistant that translates between formal logic and natural language.",
            messages=[
                {"role": "user", "content": prompt}
            ]
        )

        # Extract text from Claude response
        formal_text = "".join(
            blk.text for blk in response.content if getattr(blk, "type", None) == "text"
        ).strip()

        gpt_formal_s_list.append(formal_text)

    # Note: 'gpt_natural_s' mirrors your original code (last value from loop)
    results[entry] = {
        "natural_s": natural_s,
        "natural_p": natural_p,
        "formal_s": formal_s,
        "gpt_natural_s": gpt_natural_s,
        "gpt_natural_s_list": gpt_natural_s_list,
        "gpt_formal_s_list": gpt_formal_s_list
    }

# Save results to a JSON file
with open('claude_translations_latex_10_roundtrip.json', 'w') as out_file:
    json.dump(results, out_file, indent=4)


Processing entries:  49%|████▉     | 119/244 [29:22<30:51, 14.81s/it] 


KeyboardInterrupt: 

In [ ]:
# # parse each entities gpt_formal_s_list and remove  ```isabelle\n AND  \n```
# with open('claude_translations_latex_1_formal.json', 'r') as out_file:
#     my_json = json.load(out_file)

# for entry in my_json:
#     formal_s = my_json[entry]["gpt_formal_s"]
#     # Remove ```isabelle\n at the start and \n``` at the end if they exist
#     if formal_s.startswith("```isabelle"):
#         # replace with theorem:\n
#         formal_s = formal_s.replace("```isabelle\n", "theorem:\n ").lstrip()
#     else:
#         # add theorem:\n at the start
#         formal_s = "theorem:\n " + formal_s.lstrip()
#     if formal_s.endswith("```"):
#         formal_s = formal_s[:-len("```")].rstrip()
#     my_json[entry]["gpt_formal_s"] = formal_s

# # Save cleaned results to a JSON file
# with open('claude_translations_latex_1_formal_cleaned.json', 'w') as out_file:
#     json.dump(my_json, out_file, indent=4)

In [ ]:
# -*- coding: utf-8 -*-
import os
import json
from tqdm import tqdm
import httpx
from anthropic import Anthropic

# --- Claude 3.7 Sonnet client (disable SSL verification only if you must) ---
# ⚠️  In production, keep SSL verification enabled.
client = Anthropic(
    api_key=" ",
    http_client=httpx.Client(verify=False)
)

with open('claude_translations_latex_1_formal.json', 'r') as vd:
    valid_json = json.load(vd)

# Collect results from LLM in dictionary
results = {}

# Use tqdm for progress bar
for entry in tqdm(valid_json, desc="Processing entries"):
    natural_s = valid_json[entry]["natural_s"]
    natural_p = valid_json[entry]["natural_p"]
    formal_s = valid_json[entry]["formal_s"]

    # Prompt (identical)
    prompt = f"""
            Translate the following natural language statement into 10 clear Isabelle/HOL statements that conveys the exact same logical meaning.
            
            You convert natural language math statements into Isabelle/HOL.

            Example 1 (NL):
            For every natural number n, n + 0 = n.
            Example 1 (Isabelle):
            ∀n::nat. n + 0 = n

            Example 2 (NL):
            If A is a subset of B and B is a subset of C, then A is a subset of C.
            Example 2 (Isabelle):
            ⋀A B C. A ⊆ B ⟹ B ⊆ C ⟹ A ⊆ C

            Example 3 (NL):
            There exists a real x such that x² = 2.
            Example 3 (Isabelle):
            ∃x::real. x^2 = 2

            Now, generate a numbered list of 10 unique Isabelle/HOL statements that conveys the exact same logical meaning as the natural language statement, without any additional commentary or explanation.
            Natural Language Statement:
            {natural_s}

            Formal Statements:
            1.
            """

    # ---- Claude 3.7 Sonnet call ----
    response = client.messages.create(
        model="claude-3-7-sonnet-latest",
        max_tokens=5000,
        temperature=0.0,
        system="You are a helpful assistant that translates between formal logic and natural language.",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    # Extract text parts from Claude response
    claude_output = "".join(
        blk.text for blk in response.content if getattr(blk, "type", None) == "text"
    ).strip()

    # Keep entire entry but add gpt_formal_s (Claude output)
    results[entry] = {
        "natural_s": natural_s,
        "natural_p": natural_p,
        "formal_s": formal_s,
        "gpt_formal_s": claude_output
    }

# Save results to a JSON file
with open('claude_translations_latex_10_few_shot.json', 'w') as out_file:
    json.dump(results, out_file, indent=4)


Processing entries:   1%|          | 3/244 [00:22<29:39,  7.38s/it]


KeyboardInterrupt: 

In [ ]:
# -*- coding: utf-8 -*-
import os
import json
from tqdm import tqdm
import httpx
from anthropic import Anthropic

# --- Claude 3.7 Sonnet client (disable SSL verification only if you must) ---
# ⚠️  In production, keep SSL verification enabled.
client = Anthropic(
    api_key=" ",
    http_client=httpx.Client(verify=False)
)

with open('gpt_translations_latex.json', 'r') as vd:
    valid_json = json.load(vd)

# Collect results from LLM in dictionary
results = {}

# Use tqdm for progress bar
for entry in tqdm(valid_json, desc="Processing entries"):
    natural_s = valid_json[entry]["natural_s"]
    natural_p = valid_json[entry]["natural_p"]
    formal_s = valid_json[entry]["formal_s"]
    gpt_natural_s = valid_json[entry]["gpt_natural_s"]

    # Prompt (identical)
    prompt = f"""
            Translate the following natural language statement into 10 clear Isabelle/HOL statements that conveys the exact same logical meaning.

            Generate a numbered list of 10 unique Isabelle/HOL statements that conveys the exact same logical meaning as the natural language statement, without any additional commentary or explanation.
            
            Natural Language Statement:
            {gpt_natural_s}

            Formal Statements:
            1.
            """

    # ---- Claude 3.7 Sonnet call ----
    response = client.messages.create(
        model="claude-3-7-sonnet-latest",
        max_tokens=5000,
        temperature=0.0,
        system="You are a helpful assistant that translates between formal logic and natural language.",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    # Extract text parts from Claude response
    claude_output = "".join(
        blk.text for blk in response.content if getattr(blk, "type", None) == "text"
    ).strip()

    # Keep entire entry but add gpt_formal_s (Claude output)
    results[entry] = {
        "natural_s": natural_s,
        "natural_p": natural_p,
        "formal_s": formal_s,
        "gpt_formal_s": claude_output
    }

# Save results to a JSON file
with open('claude_translations_latex_round_all_CROSSVALIDATE.json', 'w') as out_file:
    json.dump(results, out_file, indent=4)


Processing entries: 100%|██████████| 244/244 [28:03<00:00,  6.90s/it]


In [14]:
import json
with open('claude_translations_latex_round_all_CROSSVALIDATE.json', 'r') as out_file:
    my_json = json.load(out_file)


for entry in my_json:
    # Try to split based on "\item" after removing "\begin{enumerate}" and "\end{enumerate}"
    # Otherwise we need to split based on new lines
    
    items = my_json[entry]["gpt_formal_s"].split("\n")
    # Remove leading numbers after splitting on dot or parenthesis
    items = [item.split('.', 1)[-1] if '.' in item else item.split(')', 1)[-1] for item in items]

    # Add "theorem:\n " at the start of all if not present
    # Loop through items and add "theorem:\n " if not present
    for i in range(len(items)):
        if not items[i].strip().startswith("theorem:"):
            items[i] = "theorem:\n " + items[i].strip()
        else:
            items[i] = items[i].strip()
            
    items = [item.strip() for item in items if item.strip()]


    my_json[entry]["gpt_formal_s_list"] = items
    
   
with open('claude_translations_latex_round_all_CROSSVALIDATE_formatted.json', 'w') as out_file:
    json.dump(my_json, out_file, indent=4)
